# 📥 Phase 2 — Data Collection
**StockGro Capstone | NSE Stock Analysis**

Covers: yfinance download → raw CSV → quality report → summary statistics

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from pathlib import Path
from statsmodels.tsa.stattools import adfuller

# ── CONFIG ──────────────────────────────────
BASE       = Path('.')          # project root
RAW_DIR    = BASE / 'data/raw'
PROC_DIR   = BASE / 'data/processed'
OUT_EDA    = BASE / 'outputs/charts/eda'
RPT_DIR    = BASE / 'outputs/reports'
for d in [RAW_DIR, PROC_DIR, OUT_EDA, RPT_DIR]: d.mkdir(parents=True, exist_ok=True)

FULL_START  = '2021-01-01'
FULL_END    = '2025-06-30'
TRAIN_END   = '2025-06-30'
TEST_START  = '2025-07-01'

TICKERS = {
    'HDFCBANK.NS':'HDFC Bank','ICICIBANK.NS':'ICICI Bank',
    'INFY.NS':'Infosys','TCS.NS':'TCS',
    'SUNPHARMA.NS':'Sun Pharma','DRREDDY.NS':"Dr Reddy's",
    'HINDUNILVR.NS':'HUL','ITC.NS':'ITC',
    'MARUTI.NS':'Maruti','TATAMOTORS.NS':'Tata Motors',
}
print('✅ Configuration loaded')

✅ Configuration loaded


## Step 1: Download Data from Yahoo Finance

In [2]:
# ── Download or load from disk ───────────────────────────────
stock_data = {}
download_log = []

for ticker, name in TICKERS.items():
    csv_path = RAW_DIR / f'{ticker}.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path, index_col='Date', parse_dates=True)
        print(f'  📂 {ticker:<18} loaded from disk ({len(df)} rows)')
    else:
        print(f'  🌐 Downloading {ticker} ...')
        df = yf.download(ticker, start=FULL_START, end=FULL_END,
                         progress=False, auto_adjust=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if not df.empty:
            df.to_csv(csv_path)
            print(f'  ✅ {ticker:<18} {len(df)} rows saved')
        else:
            print(f'  ❌ {ticker} failed')
            continue
    stock_data[ticker] = df

print(f'\n✅ Loaded {len(stock_data)}/{len(TICKERS)} stocks')

  🌐 Downloading HDFCBANK.NS ...


$HDFCBANK.NS: possibly delisted; no price data found  (1d 2021-01-01 -> 2025-06-30)

1 Failed download:
['HDFCBANK.NS']: possibly delisted; no price data found  (1d 2021-01-01 -> 2025-06-30)


  ❌ HDFCBANK.NS failed
  🌐 Downloading ICICIBANK.NS ...
  ✅ ICICIBANK.NS       1109 rows saved
  🌐 Downloading INFY.NS ...
  ✅ INFY.NS            1109 rows saved
  🌐 Downloading TCS.NS ...
  ✅ TCS.NS             1109 rows saved
  🌐 Downloading SUNPHARMA.NS ...
  ✅ SUNPHARMA.NS       1109 rows saved
  🌐 Downloading DRREDDY.NS ...
  ✅ DRREDDY.NS         1109 rows saved
  🌐 Downloading HINDUNILVR.NS ...
  ✅ HINDUNILVR.NS      1109 rows saved
  🌐 Downloading ITC.NS ...
  ✅ ITC.NS             1109 rows saved
  🌐 Downloading MARUTI.NS ...
  ✅ MARUTI.NS          1109 rows saved
  🌐 Downloading TATAMOTORS.NS ...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


  ❌ TATAMOTORS.NS failed

✅ Loaded 8/10 stocks


## Step 2: Data Quality Report

In [3]:
qr_rows = []
for t, df in stock_data.items():
    c = df['Close']; r = c.pct_change().dropna()
    bdays = pd.bdate_range(FULL_START, FULL_END)
    missing_bdays = len(bdays.difference(df.index))
    qr_rows.append({
        'Ticker': t, 'Company': TICKERS[t],
        'Rows': len(df), 'Missing_BDays': missing_bdays,
        'Null_Values': int(df.isnull().sum().sum()),
        'Zero_Volume': int((df['Volume']==0).sum()),
        'Price_Anomalies(>20%)': int((r.abs()>0.20).sum()),
        'Coverage_%': round(len(df)/1176*100,1),
        'Status': '✅ PASS' if df.isnull().sum().sum()==0 else '⚠️ WARN'
    })
qr = pd.DataFrame(qr_rows)
qr.to_csv(RPT_DIR/'data_quality_report.csv', index=False)
display(qr)
print('\n✅ Quality report saved')

,Ticker,Company,Rows,Missing_BDays,Null_Values,Zero_Volume,Price_Anomalies(>20%),Coverage_%,Status
0,ICICIBANK.NS,ICICI Bank,1109,64,0,1,0,94.3,✅ PASS
1,INFY.NS,Infosys,1109,64,0,1,0,94.3,✅ PASS
2,TCS.NS,TCS,1109,64,0,1,0,94.3,✅ PASS
3,SUNPHARMA.NS,Sun Pharma,1109,64,0,1,0,94.3,✅ PASS
4,DRREDDY.NS,Dr Reddy's,1109,64,0,1,0,94.3,✅ PASS
5,HINDUNILVR.NS,HUL,1109,64,0,1,0,94.3,✅ PASS
6,ITC.NS,ITC,1109,64,0,0,0,94.3,✅ PASS
7,MARUTI.NS,Maruti,1109,64,0,1,0,94.3,✅ PASS



✅ Quality report saved


## Step 3: Summary Statistics

In [4]:
ss_rows = []
for t, df in stock_data.items():
    c = df['Close'].dropna()
    r = c.pct_change().dropna()
    ss_rows.append({
        'Ticker': t, 'Company': TICKERS[t],
        'Price_Min': round(c.min(),2), 'Price_Max': round(c.max(),2),
        'Price_Mean': round(c.mean(),2), 'Price_Std': round(c.std(),2),
        'Total_Ret%': round((c.iloc[-1]/c.iloc[0]-1)*100,2),
        'Ann_Ret%': round(r.mean()*252*100,2),
        'Ann_Vol%': round(r.std()*np.sqrt(252)*100,2),
        'Sharpe': round((r.mean()*252-0.06)/(r.std()*np.sqrt(252)),3),
        'Skewness': round(float(r.skew()),3),
        'Kurtosis': round(float(r.kurt()),3),
        'Max_1D_Gain%': round(r.max()*100,2),
        'Max_1D_Loss%': round(r.min()*100,2),
    })
ss = pd.DataFrame(ss_rows)
ss.to_csv(RPT_DIR/'summary_statistics.csv', index=False)
display(ss)
print('\n✅ Summary statistics saved')

,Ticker,Company,Price_Min,Price_Max,Price_Mean,Price_Std,Total_Ret%,Ann_Ret%,Ann_Vol%,Sharpe,Skewness,Kurtosis,Max_1D_Gain%,Max_1D_Loss%
0,ICICIBANK.NS,ICICI Bank,522.35,1462.20,939.52,237.53,177.19,25.79,22.87,0.865,0.770,8.751,12.44,-7.63
1,INFY.NS,Infosys,1223.40,1999.70,1574.50,189.87,27.57,8.53,24.42,0.103,-0.214,3.792,7.93,-9.42
2,TCS.NS,TCS,2894.30,4553.75,3569.23,376.71,17.51,5.90,21.12,-0.005,0.131,2.504,6.63,-6.35
3,SUNPHARMA.NS,Sun Pharma,564.35,1948.70,1151.10,400.60,183.04,25.94,21.38,0.933,0.639,3.861,10.09,-4.55
4,DRREDDY.NS,Dr Reddy's,743.77,1412.49,1050.58,170.28,24.11,7.38,22.18,0.062,-0.450,5.181,8.13,-10.49
5,HINDUNILVR.NS,HUL,1943.95,3028.55,2458.14,180.67,-3.38,1.28,20.32,-0.232,0.065,3.014,5.96,-7.40
6,ITC.NS,ITC,192.20,503.36,343.15,99.48,103.31,18.35,21.06,0.587,0.174,3.053,6.83,-6.32
7,MARUTI.NS,Maruti,6455.65,13495.60,9558.29,1980.61,64.37,14.10,23.70,0.342,0.349,3.278,7.29,-6.60



✅ Summary statistics saved


## Step 4: EDA Visualisations

In [5]:
# Load pre-generated charts
from IPython.display import Image, display as disp
charts = sorted(OUT_EDA.glob('*.png'))
for ch in charts:
    print(f'📊 {ch.name}')
    disp(Image(filename=str(ch), width=900))